# YOLO12-Small + ECA-Net — CH-RDD2022 (Kaggle)

Notebook ini melatih YOLO12-Small dengan Efficient Channel Attention (ECA-Net) pada dataset CH-RDD2022. Hasil training, checkpoint, konfigurasi, metrik, dan prediksi test dikemas ke ZIP di `/kaggle/working`.

Notebook meng-clone branch `modification-yolo` dari repository GitHub ini, memasangnya secara editable, lalu memakai implementasi `ECAAttention` dan `yolo12-eca.yaml` langsung dari hasil clone tersebut.

Hyperparameter mengikuti Tabel 3 dari *BL-YOLOv8: An Improved Road Defect Detection Model Based on YOLOv8*: SGD, learning rate 0.01, momentum 0.937, weight decay 0.0005, batch 64, image size 640, dan 160 epoch.

Sebelum menjalankan, pilih **Accelerator: GPU** pada Kaggle. Aktifkan Internet bila `ultralytics` belum tersedia di environment Kaggle.

In [ ]:
# 1. Clone repository modifikasi, install package editable, lalu cek environment Kaggle

import sys
import json
import platform
import subprocess
import zipfile
from pathlib import Path

WORKDIR = Path('/kaggle/working')
REPO_URL = 'https://github.com/danial2015/yolo-aceh-rdd2022.git'
REPO_BRANCH = 'modification-yolo'
REPO_DIR = WORKDIR / 'yolo-aceh-rdd2022'
REPO_METADATA = WORKDIR / 'repository_revision.txt'

def log_section(title: str) -> None:
    line = '=' * 90
    print(f'\n{line}\n{title}\n{line}')
log_section('CLONE AND INSTALL MODIFIED REPOSITORY')
if REPO_DIR.exists():
    print(f'Repository sudah ada. Menyinkronkan {REPO_BRANCH} ...')
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--depth', '1', 'origin', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--force', REPO_BRANCH], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'reset', '--hard', f'origin/{REPO_BRANCH}'], check=True)
else:
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', REPO_BRANCH, REPO_URL, str(REPO_DIR)], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO_DIR)], check=True)
REPO_COMMIT = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
REPO_METADATA.write_text(
    f'repository={REPO_URL}\nbranch={REPO_BRANCH}\ncommit={REPO_COMMIT}\n', encoding='utf-8'
)
print(f'Repository : {REPO_DIR}')
print(f'Branch     : {REPO_BRANCH}')
print(f'Commit     : {REPO_COMMIT}')

# Import dilakukan setelah package versi repository selesai dipasang.
sys.path.insert(0, str(REPO_DIR))
import torch
import ultralytics

log_section('KAGGLE ENVIRONMENT')
print(f'Python       : {platform.python_version()}')
print(f'PyTorch      : {torch.__version__}')
print(f'Ultralytics  : {ultralytics.__version__}')
print(f'Ultralytics path: {Path(ultralytics.__file__).resolve()}')
print(f'CUDA ready   : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU          : {torch.cuda.get_device_name(0)}')
    DEVICE = 0
else:
    print('WARNING: GPU tidak ditemukan. Training akan sangat lambat pada CPU.')
    DEVICE = 'cpu'


In [ ]:
# 2. Konfigurasi dataset dan parameter training
DATA_ROOT = Path('/kaggle/input/datasets/danialalfayyadh/ch-rdd2022/datasets-china-split')
DATA_YAML = WORKDIR / 'ch_rdd2022.yaml'
MODEL_YAML = REPO_DIR / 'ultralytics/cfg/models/12/yolo12-eca.yaml'
ECA_SOURCE_FILES = (
    REPO_DIR / 'ultralytics/nn/modules/conv.py',
    REPO_DIR / 'ultralytics/nn/modules/__init__.py',
    REPO_DIR / 'ultralytics/nn/tasks.py',
)
RUNS_DIR = WORKDIR / 'runs'

# Hyperparameter Tabel 3 paper BL-YOLOv8.
EPOCHS = 160
IMGSZ = 640
BATCH = 64
LR0 = 0.01
MOMENTUM = 0.937
WEIGHT_DECAY = 0.0005
OPTIMIZER = 'SGD'
PATIENCE = 0  # Menonaktifkan early stopping agar seluruh 160 epoch dijalankan.
WORKERS = 2
SEED = 42
EXPERIMENT_NAME = 'yolo12s_eca_ch_rdd2022_pretrained'

print('Paper setting: SGD | lr0=0.01 | momentum=0.937 | weight_decay=0.0005 | batch=64 | imgsz=640 | epochs=160')
print('Catatan: batch 64 digunakan paper pada RTX 3090 24 GB. Jika GPU Kaggle kehabisan memori, turunkan batch; hasilnya tidak lagi identik dengan setup paper.')

DATA_YAML.write_text(
    f'''path: {DATA_ROOT}
train: train/images
val: val/images
test: test/images

nc: 5
names:
  0: D00
  1: D10
  2: D20
  3: D40
  4: Repair
''',
    encoding='utf-8',
)

log_section('DATASET CONFIGURATION')
print(DATA_YAML.read_text())
assert DATA_ROOT.exists(), f'Dataset path tidak ditemukan: {DATA_ROOT}'

image_suffixes = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
for split in ('train', 'val', 'test'):
    image_dir = DATA_ROOT / split / 'images'
    label_dir = DATA_ROOT / split / 'labels'
    image_count = sum(path.suffix.lower() in image_suffixes for path in image_dir.rglob('*')) if image_dir.exists() else 0
    label_count = len(list(label_dir.glob('*.txt'))) if label_dir.exists() else 0
    print(f'{split:>5}: {image_count:>6} images | {label_count:>6} label files | {image_dir}')
    if split in {'train', 'val'}:
        assert image_count > 0, f'Tidak ada gambar pada {image_dir}'


In [ ]:
# 3. Verifikasi bahwa ECA-Net dan YAML dipakai langsung dari repository yang di-clone
from ultralytics.nn.modules import ECAAttention

assert MODEL_YAML.exists(), f'YAML ECA tidak ditemukan di repository: {MODEL_YAML}'
assert all(path.exists() for path in ECA_SOURCE_FILES), 'Source file ECA pada repository tidak lengkap.'
log_section('REPOSITORY ECA IMPLEMENTATION')
print(f'ECA module : {ECAAttention.__module__}.{ECAAttention.__name__}')
print(f'Model YAML : {MODEL_YAML}')
print(f'Source root: {REPO_DIR}')
print(f'Git commit : {REPO_COMMIT}')


In [ ]:
# 4. Cetak konfigurasi YOLO12-Small + ECA yang berada di repository hasil clone
log_section('YOLO12S + ECA MODEL YAML FROM REPOSITORY')
print(MODEL_YAML.read_text(encoding='utf-8'))


In [ ]:
# 5. Cetak perbandingan model standar vs. model ECA agar log Kaggle ringkas dan mudah dibaca
from ultralytics.nn.tasks import DetectionModel

baseline_model = DetectionModel('yolo12s.yaml', nc=5, verbose=False)
eca_model = DetectionModel(str(MODEL_YAML), nc=5, verbose=False)
baseline_params = sum(parameter.numel() for parameter in baseline_model.parameters())
eca_params = sum(parameter.numel() for parameter in eca_model.parameters())
eca_layers = [
    (layer.i, layer.conv.kernel_size[0], layer.conv.padding[0])
    for layer in eca_model.model
    if isinstance(layer, ECAAttention)
]

log_section('MODEL COMPARISON')
print(f'YOLO12-Small parameters      : {baseline_params:,}')
print(f'YOLO12-Small + ECA parameters: {eca_params:,}')
print(f'Additional parameters         : {eca_params - baseline_params:,}')
print(f'ECA layers (index, kernel, pad): {eca_layers}')
print('Expected ECA layout           : P3 -> ECA(k=5), P4 -> ECA(k=5), P5 -> ECA(k=5)')

# Lepaskan model perbandingan dari memori GPU sebelum training.
del baseline_model, eca_model
if torch.cuda.is_available():
    torch.cuda.empty_cache()


In [ ]:
# 6. Inisialisasi YOLO12-Small + ECA dari bobot pretrained YOLO12-Small resmi
# Tiga ECA baru tidak memiliki pasangan pada model sumber sehingga tetap diinisialisasi dan dipelajari saat training.
import re
from ultralytics import YOLO

PRETRAINED_WEIGHTS = 'yolo12s.pt'

def target_layer_index(source_index: int) -> int:
    """Map YOLO12s layer indices to YOLO12s-ECA after ECA is inserted after source layers 4, 6, and 8."""
    return source_index + int(source_index >= 5) + int(source_index >= 7) + int(source_index >= 9)

def remap_yolo12s_weights(source_state: dict, target_state: dict) -> dict:
    """Transfer only source tensors whose remapped names and shapes match the ECA model."""
    transferred = {}
    pattern = re.compile(r'^model\.(\d+)(\..+)$')
    for source_key, source_tensor in source_state.items():
        match = pattern.match(source_key)
        if match is None:
            continue
        target_key = f'model.{target_layer_index(int(match.group(1)))}{match.group(2)}'
        if target_key in target_state and target_state[target_key].shape == source_tensor.shape:
            transferred[target_key] = source_tensor
    return transferred

log_section('PRETRAINED WEIGHT TRANSFER')
print(f'Downloading/loading source checkpoint: {PRETRAINED_WEIGHTS}')
model = YOLO(str(MODEL_YAML))
source_model = YOLO(PRETRAINED_WEIGHTS).model.float()
target_state = model.model.state_dict()
transferred_state = remap_yolo12s_weights(source_model.state_dict(), target_state)
incompatible = model.model.load_state_dict(transferred_state, strict=False)
PRETRAINED_REPORT = {
    'source_weights': PRETRAINED_WEIGHTS,
    'transferred_tensors': len(transferred_state),
    'target_tensors': len(target_state),
    'uninitialized_tensors': len(incompatible.missing_keys),
}

# Inform Model.train() bahwa model ini membawa bobot yang harus ditransfer ke head 5 kelas.
model.ckpt = {'model': model.model}
del source_model, target_state, transferred_state
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print(f"Transferred tensors : {PRETRAINED_REPORT['transferred_tensors']}/{PRETRAINED_REPORT['target_tensors']}")
print(f"Uninitialized tensors: {PRETRAINED_REPORT['uninitialized_tensors']} (ECA and incompatible detection-class tensors)")
print('Training akan membuat head 5 kelas dan hanya mempertahankan tensor pretrained yang kompatibel.')


In [ ]:
# 7. Training. Ultralytics akan menampilkan metrik per epoch pada log Kaggle dan menyimpan grafik otomatis.

log_section('TRAINING STARTED')
print(f'Experiment : {EXPERIMENT_NAME}')
print(f'Epochs     : {EPOCHS}')
print(f'Image size : {IMGSZ}')
print(f'Batch      : {BATCH}')
print(f'Optimizer  : {OPTIMIZER}')
print(f'LR0        : {LR0}')
print(f'Momentum   : {MOMENTUM}')
print(f'Weight decay: {WEIGHT_DECAY}')
print(f'Early stop : disabled (patience={PATIENCE})')
print(f'Device     : {DEVICE}')

model.train(
    data=str(DATA_YAML),
    epochs=EPOCHS,
    imgsz=IMGSZ,
    batch=BATCH,
    device=DEVICE,
    workers=WORKERS,
    project=str(RUNS_DIR),
    name=EXPERIMENT_NAME,
    exist_ok=True,
    pretrained=True,
    optimizer=OPTIMIZER,
    lr0=LR0,
    momentum=MOMENTUM,
    weight_decay=WEIGHT_DECAY,
    cos_lr=False,
    patience=PATIENCE,
    seed=SEED,
    plots=True,
    verbose=True,
)

RUN_DIR = Path(model.trainer.save_dir)
BEST_PT = Path(model.trainer.best)
LAST_PT = Path(model.trainer.last)
print(f'\nRun directory: {RUN_DIR}')
print(f'Best weights : {BEST_PT}')
print(f'Last weights : {LAST_PT}')


In [ ]:
# 8. Validasi best checkpoint dan evaluasi/prediksi test
log_section('BEST CHECKPOINT EVALUATION')
best_model = YOLO(str(BEST_PT))
val_metrics = best_model.val(
    data=str(DATA_YAML),
    split='val',
    imgsz=IMGSZ,
    batch=16,
    device=DEVICE,
    project=str(RUNS_DIR),
    name=f'{EXPERIMENT_NAME}_val',
    exist_ok=True,
    plots=True,
)
print(f'Validation mAP50-95: {val_metrics.box.map:.4f}')
print(f'Validation mAP50   : {val_metrics.box.map50:.4f}')

test_label_dir = DATA_ROOT / 'test' / 'labels'
test_has_labels = test_label_dir.exists() and any(test_label_dir.glob('*.txt'))
if test_has_labels:
    test_metrics = best_model.val(
        data=str(DATA_YAML),
        split='test',
        imgsz=IMGSZ,
        batch=16,
        device=DEVICE,
        project=str(RUNS_DIR),
        name=f'{EXPERIMENT_NAME}_test',
        exist_ok=True,
        plots=True,
    )
    print(f'Test mAP50-95: {test_metrics.box.map:.4f}')
    TEST_OUTPUT_DIR = Path(test_metrics.save_dir)
else:
    print('Test labels tidak ditemukan; menjalankan prediksi test tanpa menghitung mAP.')
    best_model.predict(
        source=str(DATA_ROOT / 'test' / 'images'),
        imgsz=IMGSZ,
        device=DEVICE,
        conf=0.25,
        save=True,
        save_txt=True,
        project=str(RUNS_DIR),
        name=f'{EXPERIMENT_NAME}_test_predictions',
        exist_ok=True,
        verbose=True,
    )
    TEST_OUTPUT_DIR = RUNS_DIR / f'{EXPERIMENT_NAME}_test_predictions'

print(f'Test output: {TEST_OUTPUT_DIR}')


In [ ]:
# 9. Simpan hasil ke satu ZIP yang siap diunduh dari panel Output Kaggle
ZIP_PATH = WORKDIR / f'{EXPERIMENT_NAME}_results.zip'
RUN_CONFIG = WORKDIR / f'{EXPERIMENT_NAME}_config.json'
RUN_CONFIG.write_text(
    json.dumps(
        {
            'dataset_root': str(DATA_ROOT),
            'repository_url': REPO_URL,
            'repository_branch': REPO_BRANCH,
            'repository_commit': REPO_COMMIT,
            'pretrained_transfer': PRETRAINED_REPORT,
            'model_yaml': str(MODEL_YAML),
            'epochs': EPOCHS,
            'imgsz': IMGSZ,
            'batch': BATCH,
            'optimizer': OPTIMIZER,
            'lr0': LR0,
            'momentum': MOMENTUM,
            'weight_decay': WEIGHT_DECAY,
            'patience': PATIENCE,
            'device': str(DEVICE),
            'seed': SEED,
            'best_checkpoint': str(BEST_PT),
            'last_checkpoint': str(LAST_PT),
        },
        indent=2,
    ),
    encoding='utf-8',
)

def add_to_zip(archive: zipfile.ZipFile, path: Path) -> int:
    """Add a file or directory to the archive and return the number of files added."""
    if not path.exists():
        return 0
    files = [path] if path.is_file() else [item for item in path.rglob('*') if item.is_file()]
    for file_path in files:
        archive.write(file_path, file_path.relative_to(WORKDIR))
    return len(files)

log_section('CREATE RESULTS ZIP')
with zipfile.ZipFile(ZIP_PATH, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    file_count = 0
    for artifact in (RUN_DIR, TEST_OUTPUT_DIR, DATA_YAML, MODEL_YAML, *ECA_SOURCE_FILES, REPO_METADATA, RUN_CONFIG):
        file_count += add_to_zip(archive, Path(artifact))

print(f'ZIP created : {ZIP_PATH}')
print(f'ZIP size    : {ZIP_PATH.stat().st_size / (1024 ** 2):.2f} MB')
print(f'Files added : {file_count}')
print('Download file tersebut dari panel Output / Files di sisi kanan Kaggle.')

from IPython.display import FileLink, display
display(FileLink(ZIP_PATH))
